# Task 3: Logistic Regression
Logistic Regression gives each word a weight.
A positive weight pushes the message to spam, a negative weight to ham.
It adds the weights of all the words in a message and decides.
We try it on our three text versions and choose the best settings.

## 1. Load the data

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix, classification_report
train = pd.read_csv('../data/train_prep_v2.csv')
test = pd.read_csv('../data/test_prep_v2.csv')
y_train = (train['label'] == 'spam').astype(int)
y_test = (test['label'] == 'spam').astype(int)
print('Train spam:', y_train.sum(), '| Test spam:', y_test.sum(), 'of', len(y_test))

Train spam: 501 | Test spam: 125 of 1026


## 2. Choosing the settings in a fair way
We use cross-validation on the train data, like in the Naive Bayes notebook.
The test data is used one time only, at the end.

We try two settings:
- C: how free the model is. A big C lets the weights grow,
  a small C keeps them small so the model stays simple.
- class_weight: our data has much more ham than spam.
  With "balanced", the model gives more importance to the spam messages,
  because they are fewer.

In [2]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for col in ['clean_text', 'stop_text', 'stem_text']:
    for C in [1, 10, 100]:
        for weight in [None, 'balanced']:
            model = LogisticRegression(C=C, class_weight=weight,
                                       max_iter=1000, random_state=42)
            pipe = make_pipeline(TfidfVectorizer(token_pattern=r'\S+'), model)
            score = cross_val_score(pipe, train[col], y_train, cv=cv, scoring='f1')
            rows.append([col, C, str(weight), round(100 * score.mean(), 1)])
result = pd.DataFrame(rows, columns=['text', 'C', 'class_weight', 'cv_f1'])
print(result.sort_values('cv_f1', ascending=False).to_string(index=False))

      text   C class_weight  cv_f1
 stem_text 100     balanced   96.0
clean_text  10     balanced   95.9
 stop_text  10     balanced   95.9
 stem_text  10     balanced   95.8
clean_text 100     balanced   95.7
 stop_text 100     balanced   95.6
clean_text   1     balanced   95.5
 stem_text   1     balanced   95.5
 stop_text   1     balanced   95.4
 stop_text 100         None   95.2
 stem_text 100         None   95.2
clean_text 100         None   95.0
 stop_text  10         None   94.2
 stem_text  10         None   94.2
clean_text  10         None   93.9
 stem_text   1         None   90.3
 stop_text   1         None   90.0
clean_text   1         None   88.8


## 3. Train the final model and test it
We use the best settings from the cross-validation:
stem_text, C=100, class_weight='balanced'.

In [3]:
best_col = 'stem_text'
vec = TfidfVectorizer(token_pattern=r'\S+')
X_train = vec.fit_transform(train[best_col])
X_test = vec.transform(test[best_col])
model = LogisticRegression(C=100, class_weight='balanced',
                           max_iter=1000, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Accuracy :', round(100 * accuracy_score(y_test, pred), 1))
print('Precision:', round(100 * precision_score(y_test, pred), 1))
print('Recall   :', round(100 * recall_score(y_test, pred), 1))
print('F1       :', round(100 * f1_score(y_test, pred), 1))
cm = confusion_matrix(y_test, pred)
print(pd.DataFrame(cm, index=['true ham', 'true spam'],
                   columns=['said ham', 'said spam']))
print(classification_report(y_test, pred, target_names=['ham', 'spam'], digits=3))

Accuracy : 99.1
Precision: 97.5
Recall   : 95.2
F1       : 96.4
           said ham  said spam
true ham        898          3
true spam         6        119
              precision    recall  f1-score   support

         ham      0.993     0.997     0.995       901
        spam      0.975     0.952     0.964       125

    accuracy                          0.991      1026
   macro avg      0.984     0.974     0.979      1026
weighted avg      0.991     0.991     0.991      1026



## 4. The words with the biggest weights
Logistic Regression gives every word a weight.
We can read the weights and see which words push to spam
and which words push to ham.
This is useful: Naive Bayes cannot show this so clearly.

In [4]:
weights = pd.Series(model.coef_[0], index=vec.get_feature_names_out())
weights = weights.sort_values()
print('Words that push to SPAM:')
print(weights.tail(12).round(2))
print('Words that push to HAM:')
print(weights.head(12).round(2))

Words that push to SPAM:
tone           7.05
won            7.30
sexi           7.31
146tf150p      7.33
mobil          7.59
text           7.61
repli          8.40
txt           10.59
pounds        13.11
pence         13.37
url           16.86
longnumber    29.01
dtype: float64
Words that push to HAM:
hiddeninfo   -6.80
i            -6.35
my           -5.63
that         -4.85
happyface    -4.62
me           -4.58
it           -4.38
im           -4.06
at           -3.51
ok           -3.50
bad          -3.30
when         -3.26
dtype: float64


## 5. Save the predictions
Abdullah will compare our two models, so we save what our model said.

In [5]:
out = test[['label', 'text']].copy()
out['pred'] = np.where(pred == 1, 'spam', 'ham')
out['prob_spam'] = model.predict_proba(X_test)[:, 1].round(4)
out.to_csv('../data/pred_lr.csv', index=False)
print(out['pred'].value_counts().to_dict())
out.head()

{'ham': 904, 'spam': 122}


,label,text,pred,prob_spam
0,ham,"sir, you will receive the account no another 1...",ham,0.0153
1,ham,You at mu? You should try to figure out how mu...,ham,0.0035
2,ham,The greatest test of courage on earth is to be...,ham,0.0042
3,ham,Gud mrng dear hav a nice day,ham,0.0023
4,ham,I wonder if you'll get this text?,ham,0.0170
